# Sequence Fetch Examples

This notebook demonstrates multi-database sequence retrieval using **`run_sequence_fetch`**, the orchestrator that chains NCBI Entrez, UniProt, and PDB with automatic fallback.

- **Protein retrieval** -- Fetch a protein sequence by name and organism
- **Multi-type requests** -- Retrieve protein + CDS + structure in one call
- **Batch requests** -- Process multiple targets with adaptive batching
- **Direct ID overrides** -- Bypass name search with known accessions

## Imports

In [1]:
from bio_programming_tools import (
    SequenceFetchConfig,
    SequenceFetchInput,
    SequenceFetchRequest,
    run_sequence_fetch,
)


## API Reference

**`SequenceFetchRequest`**

| Field | Type | Default | Description |
|---|---|---|---|
| `request_id` | `Optional[str]` | `None` | Caller-supplied identifier for correlation |
| `target_name` | `str` | *(required)* | Gene, protein, or RNA name to resolve |
| `organism` | `str` | *(required)* | Organism for disambiguation |
| `sequence_types` | `List[str]` | *(required)* | Requested types: `protein`, `dna_genomic`, `dna_cds`, `rna_transcript`, `rna_premrna`, `structure` |
| `uniprot_id` | `Optional[str]` | `None` | UniProt accession override |
| `genbank_accession` | `Optional[str]` | `None` | GenBank accession override |
| `refseq_accession` | `Optional[str]` | `None` | RefSeq accession override |
| `pdb_id` | `Optional[str]` | `None` | PDB accession override |
| `gene_id` | `Optional[str]` | `None` | NCBI Gene ID override |
| `protein_id` | `Optional[str]` | `None` | NCBI protein accession override |
| `transcript_id` | `Optional[str]` | `None` | Transcript accession override |
| `genomic_coordinates` | `Optional[str]` | `None` | Coordinates as `accession:start-end:strand` |

**`SequenceFetchConfig`**

| Field | Type | Default | Description |
|---|---|---|---|
| `request_timeout_seconds` | `int` | `15` | HTTP timeout per request |
| `http_retries` | `int` | `2` | Max HTTP retries per upstream call |
| `backoff_seconds` | `float` | `1.0` | Seconds to wait between retries (doubles after each attempt) |
| `max_candidates_per_source` | `int` | `5` | Max candidates for name lookups |
| `strict_type_checks` | `bool` | `True` | Reject requests where molecule type conflicts with target (e.g. protein for an ncRNA gene) |
| `fail_on_type_mismatch` | `bool` | `True` | Treat molecule type mismatches as errors instead of warnings |
| `include_sequence_checksums` | `bool` | `True` | Include SHA256 checksums per sequence |
| `ncbi_api_key` | `Optional[str]` | `None` | Optional NCBI API key |
| `ncbi_email` | `Optional[str]` | `None` | Optional NCBI contact email |
| `user_agent` | `str` | `"bio-programming-tools/sequence-fetch-v1"` | Identifier string sent to database APIs with each request |

**`SequenceFetchResult`**

| Field | Type | Description |
|---|---|---|
| `request_id` | `str` | Correlation identifier |
| `target_name` | `str` | Original target name |
| `organism` | `str` | Original organism name |
| `status` | `Literal["success", "warning", "failed"]` | Result status |
| `fetched_sequences` | `List[FetchedSequence]` | Retrieved sequence records |
| `fetched_structures` | `List[FetchedStructure]` | Retrieved structure records |
| `resolved_ids` | `Dict[str, str]` | IDs resolved during retrieval |
| `warnings` | `List[str]` | Non-fatal warnings |
| `errors` | `List[str]` | Failure messages |

**`FetchedSequence`**

| Field | Type | Description |
|---|---|---|
| `sequence_type` | `str` | Molecule type (protein, dna_genomic, etc.) |
| `source_database` | `str` | Upstream source (ncbi, uniprot, pdb) |
| `accession` | `Optional[str]` | Source accession identifier |
| `sequence` | `str` | Retrieved sequence |
| `length` | `int` | Sequence length |

**`FetchedStructure`**

| Field | Type | Description |
|---|---|---|
| `pdb_id` | `str` | PDB accession |
| `title` | `Optional[str]` | Structure title |
| `method` | `Optional[str]` | Experimental method |
| `resolution` | `Optional[float]` | Resolution in angstroms |

## 1. Fetch a Protein Sequence

Retrieve the lac repressor protein from *E. coli* by name. The orchestrator searches UniProt first, then falls back to NCBI protein.

In [2]:
inputs = SequenceFetchInput(
    requests=[
        SequenceFetchRequest(
            target_name="lacI",
            organism="Escherichia coli",
            sequence_types=["protein"],
        )
    ]
)

output = run_sequence_fetch(inputs, SequenceFetchConfig())
result = output.results[0]

print(f"Target: {result.target_name} ({result.organism})")
print(f"Status: {result.status}")
print(f"Resolved IDs: {result.resolved_ids}")
print()

seq = result.fetched_sequences[0]
print(f"Type: {seq.sequence_type}")
print(f"Source: {seq.source_database} ({seq.accession})")
print(f"Length: {seq.length}")
print(f"Preview: {seq.sequence[:60]}...")

Target: lacI (Escherichia coli)
Status: success
Resolved IDs: {'uniprot_id': 'A0A094XSW9'}

Type: protein
Source: uniprot (A0A094XSW9)
Length: 319
Preview: MAELNYIPNRVAQQLAGKQSLLIGVATSSLALHAPSQIVAAIKSRADQLGASVVVSMVER...


## 2. Multi-Type Request

Fetch protein, CDS, and structure for TP53 in a single request. The orchestrator resolves IDs once and shares them across fetch types (e.g., UniProt ID resolved during protein fetch is reused for structure lookup).

In [3]:
inputs = SequenceFetchInput(
    requests=[
        SequenceFetchRequest(
            target_name="TP53",
            organism="Homo sapiens",
            sequence_types=["protein", "dna_cds", "structure"],
        )
    ]
)

output = run_sequence_fetch(inputs, SequenceFetchConfig())
result = output.results[0]

print(f"Target: {result.target_name} ({result.organism})")
print(f"Status: {result.status}")
print(f"Resolved IDs: {result.resolved_ids}")
print()

for seq in result.fetched_sequences:
    preview = seq.sequence[:50] + ("..." if len(seq.sequence) > 50 else "")
    print(f"  {seq.sequence_type}: {seq.source_database} ({seq.accession}), length={seq.length}")
    print(f"    {preview}")

for struct in result.fetched_structures:
    res_str = f"{struct.resolution:.1f} A" if struct.resolution else "N/A"
    print(f"  structure: {struct.pdb_id}, {struct.method}, {res_str}")
    print(f"    {struct.title}")

Target: TP53 (Homo sapiens)
Status: warning
Resolved IDs: {'uniprot_id': 'P04637', 'cds_accession': 'PX584652.1_cds_YCJ03734.1_1', 'pdb_id': '1A1U'}

  protein: uniprot (P04637), length=393
    MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDI...
  dna_cds: ncbi (PX584652.1_cds_YCJ03734.1_1), length=1182
    ATGGAGGAGCCGCAGTCAGATCCTAGCGTCGAGCCCCCTCTGAGTCAGGA...
  structure: 1A1U, SOLUTION NMR, N/A
    SOLUTION STRUCTURE DETERMINATION OF A P53 MUTANT DIMERIZATION DOMAIN, NMR, MINIMIZED AVERAGE STRUCTURE


## 3. Batch Requests

Process multiple targets in a single call.

In [4]:
inputs = SequenceFetchInput(
    requests=[
        SequenceFetchRequest(
            target_name="lacI",
            organism="Escherichia coli",
            sequence_types=["protein"],
        ),
        SequenceFetchRequest(
            target_name="insulin",
            organism="Homo sapiens",
            sequence_types=["protein"],
        ),
        SequenceFetchRequest(
            target_name="GFP",
            organism="Aequorea victoria",
            sequence_types=["protein"],
        ),
    ]
)

output = run_sequence_fetch(inputs, SequenceFetchConfig())

print(f"Completed: {output.num_completed}/{output.num_requests}")
print()

for result in output.results:
    if result.fetched_sequences:
        seq = result.fetched_sequences[0]
        print(f"{result.target_name}: {seq.source_database} ({seq.accession}), length={seq.length}")
    else:
        print(f"{result.target_name}: {result.status} -- {result.errors}")

Completed: 3/3

lacI: uniprot (A0A094XSW9), length=319
insulin: uniprot (P01308), length=110
GFP: uniprot (A0A059PIQ0), length=251


## 4. Direct ID Override

When you already have a UniProt accession or PDB ID, provide it directly to skip the name search. This is faster and avoids ambiguity.

In [5]:
inputs = SequenceFetchInput(
    requests=[
        SequenceFetchRequest(
            target_name="TP53",
            organism="Homo sapiens",
            sequence_types=["protein", "structure"],
            uniprot_id="P04637",
            pdb_id="1TSR",
        )
    ]
)

output = run_sequence_fetch(inputs, SequenceFetchConfig())
result = output.results[0]

print(f"Status: {result.status}")
print(f"Resolved IDs: {result.resolved_ids}")
print()

seq = result.fetched_sequences[0]
print(f"Protein: {seq.accession}, length={seq.length}")

struct = result.fetched_structures[0]
print(f"Structure: {struct.pdb_id}, {struct.method}")

Status: success
Resolved IDs: {'uniprot_id': 'P04637', 'pdb_id': '1TSR'}

Protein: P04637, length=393
Structure: 1TSR, X-RAY DIFFRACTION


## Export Results

Save fetched sequences to FASTA or JSON format.

In [6]:
# Export as FASTA
output.export("sequence_fetch_results", export_path="./example_output", file_format="fasta")
print("Exported to ./example_output/sequence_fetch_results.fasta")

# Export as JSON
output.export("sequence_fetch_results", export_path="./example_output", file_format="json")
print("Exported to ./example_output/sequence_fetch_results.json")

Exported to ./example_output/sequence_fetch_results.fasta
Exported to ./example_output/sequence_fetch_results.json
